# Monthly Performance Data Audit

## Purpose

This notebook tests the monthly performance files for the 2006, 2015, 2016,
and 2017 mortgage vintages.

The testing supports the simulated Enterprise Independent Testing case study
and covers:

- File and schema validation
- Population completeness
- Full-population origination matching
- Duplicate loan-month testing
- Reporting-period and loan-age validation
- Delinquency-status validation
- Performance-window coverage

## Memory safeguards

Because the computer has 8 GB of RAM:

- Process one vintage at a time
- Inspect only small samples before full testing
- Read large files in chunks
- Retain only required columns
- Save compact intermediate outputs
- Never combine all raw performance files in memory

In [1]:
from pathlib import Path
import gc
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"
vintages = [2006, 2015, 2016, 2017]

performance_files = {
    year: next(
        raw_data_dir.rglob(f"sample_perf_{year}.txt"),
        None
    )
    for year in vintages
}

structure_rows = []

for year, file_path in performance_files.items():
    if file_path is None:
        structure_rows.append({
            "vintage": year,
            "file_exists": False,
            "file_size_mb": None,
            "sample_rows_read": 0,
            "column_count": None
        })
        continue

    preview = pd.read_csv(
        file_path,
        sep="|",
        header=None,
        dtype="string",
        nrows=5
    )

    structure_rows.append({
        "vintage": year,
        "file_exists": True,
        "file_size_mb": round(
            file_path.stat().st_size / (1024**2),
            2
        ),
        "sample_rows_read": len(preview),
        "column_count": preview.shape[1]
    })

    del preview
    gc.collect()

performance_structure = pd.DataFrame(structure_rows)
performance_structure

,vintage,file_exists,file_size_mb,sample_rows_read,column_count
0,2006,True,340.70,5,35
1,2015,True,355.59,5,35
2,2016,True,349.37,5,35
3,2017,True,292.56,5,35


## Official monthly-performance schema

The performance files contain 35 pipe-delimited fields without a header row.
Readable snake_case names are assigned in the official Freddie Mac field order.

Only five rows from each vintage are loaded initially to validate field alignment.

In [2]:
performance_columns = [
    "loan_identifier",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "underwriting_defect_and_major_servicing_defect_settlement_date",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_non_interest_bearing_upb",
    "due_date_of_last_paid_installment",
    "mi_recoveries",
    "net_sales_proceeds",
    "non_mi_recoveries",
    "total_expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "miscellaneous_expenses",
    "actual_loss",
    "cumulative_modification_costs",
    "interest_rate_step_indicator",
    "payment_deferral_flag",
    "estimated_ltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_plan",
    "current_period_modification_costs",
    "current_interest_bearing_upb",
    "mortgage_insurance_cancellation_indicator",
    "servicer_name",
    "bankruptcy_cramdown_costs"
]

assert len(performance_columns) == 35

In [3]:
preview_frames = []
schema_check_rows = []

for year, file_path in performance_files.items():
    preview = pd.read_csv(
        file_path,
        sep="|",
        header=None,
        names=performance_columns,
        dtype="string",
        na_filter=False,
        nrows=5
    )

    preview.insert(0, "vintage", year)
    preview_frames.append(preview)

    periods = preview["monthly_reporting_period"].str.strip()
    loan_ages = pd.to_numeric(
        preview["loan_age"],
        errors="coerce"
    )

    schema_check_rows.append({
        "vintage": year,
        "sample_rows": len(preview),
        "performance_columns": len(performance_columns),
        "blank_loan_ids": (
            preview["loan_identifier"]
            .str.strip()
            .eq("")
            .sum()
        ),
        "invalid_period_formats": (
            ~periods.str.fullmatch(r"\d{6}")
        ).sum(),
        "delinquency_values": sorted(
            preview["current_loan_delinquency_status"]
            .str.strip()
            .unique()
            .tolist()
        ),
        "minimum_sample_loan_age": loan_ages.min(),
        "maximum_sample_loan_age": loan_ages.max()
    })

performance_schema_preview = pd.concat(
    preview_frames,
    ignore_index=True
)

performance_schema_check = pd.DataFrame(
    schema_check_rows
)

display(
    performance_schema_preview[
        [
            "vintage",
            "loan_identifier",
            "monthly_reporting_period",
            "current_actual_upb",
            "current_loan_delinquency_status",
            "loan_age",
            "remaining_months_to_legal_maturity"
        ]
    ]
)

performance_schema_check

,vintage,loan_identifier,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_legal_maturity
0,2006,F06Q10000059,200602,180000.00,00,0,360
1,2006,F06Q10000059,200603,180000.00,00,1,359
2,2006,F06Q10000059,200604,180000.00,00,2,358
3,2006,F06Q10000059,200605,180000.00,00,3,357
4,2006,F06Q10000059,200606,179000.00,00,4,356
5,2015,F15Q10000025,201503,417000.00,00,0,180
6,2015,F15Q10000025,201504,415000.00,00,1,179
7,2015,F15Q10000025,201505,413000.00,00,2,178
8,2015,F15Q10000025,201506,409000.00,00,3,177
9,2015,F15Q10000025,201507,409000.00,00,4,176


,vintage,sample_rows,performance_columns,blank_loan_ids,invalid_period_formats,delinquency_values,minimum_sample_loan_age,maximum_sample_loan_age
0,2006,5,35,0,0,[00],0,4
1,2015,5,35,0,0,[00],0,4
2,2016,5,35,0,0,[00],0,4
3,2017,5,35,0,0,[00],0,4


## Full-population performance control testing

The first full-file test uses the 2006 vintage. Only the loan identifier and
monthly reporting period are read in 100,000-row chunks.

The test evaluates:

- Population size
- Blank identifiers
- Full-population origination matching
- Reporting-period format
- Loan-month duplicates
- File sort order

If the file is confirmed to be sorted by loan and reporting period, adjacent-key
testing provides a memory-safe complete duplicate test.

In [4]:
origination_output_file = (
    project_root
    / "data"
    / "processed"
    / "originations_clean.parquet"
)

origination_id_data = pd.read_parquet(
    origination_output_file,
    columns=["loan_identifier"]
)

origination_ids = set(
    origination_id_data["loan_identifier"]
    .astype("string")
    .str.strip()
    .dropna()
)

origination_id_check = pd.Series({
    "origination_ids_loaded": len(origination_ids),
    "expected_origination_ids": 200_000,
    "matches_expected_population": (
        len(origination_ids) == 200_000
    )
})

origination_id_check

origination_ids_loaded         200000
expected_origination_ids       200000
matches_expected_population      True
dtype: object

In [5]:
from time import perf_counter

test_vintage = 2006
test_file = performance_files[test_vintage]
chunk_size = 100_000

total_rows = 0
unique_performance_ids = set()
unmatched_ids = set()

blank_loan_ids = 0
invalid_period_formats = 0
unmatched_performance_rows = 0
duplicate_loan_months = 0
sort_order_violations = 0

minimum_period = None
maximum_period = None
previous_key = None

start_time = perf_counter()

for chunk in pd.read_csv(
    test_file,
    sep="|",
    header=None,
    names=performance_columns,
    usecols=[
        "loan_identifier",
        "monthly_reporting_period"
    ],
    dtype="string",
    na_filter=False,
    chunksize=chunk_size
):
    loan_ids = (
        chunk["loan_identifier"]
        .str.strip()
    )

    periods = (
        chunk["monthly_reporting_period"]
        .str.strip()
    )

    total_rows += len(chunk)

    blank_mask = loan_ids.eq("")
    invalid_period_mask = ~periods.str.fullmatch(r"\d{6}")
    unmatched_mask = (
        ~loan_ids.isin(origination_ids)
        & ~blank_mask
    )

    blank_loan_ids += int(blank_mask.sum())
    invalid_period_formats += int(
        invalid_period_mask.sum()
    )
    unmatched_performance_rows += int(
        unmatched_mask.sum()
    )

    unique_performance_ids.update(
        loan_ids.loc[~blank_mask].unique().tolist()
    )

    unmatched_ids.update(
        loan_ids.loc[unmatched_mask].unique().tolist()
    )

    valid_periods = periods.loc[
        ~invalid_period_mask
    ]

    if not valid_periods.empty:
        chunk_minimum = valid_periods.min()
        chunk_maximum = valid_periods.max()

        minimum_period = (
            chunk_minimum
            if minimum_period is None
            else min(minimum_period, chunk_minimum)
        )

        maximum_period = (
            chunk_maximum
            if maximum_period is None
            else max(maximum_period, chunk_maximum)
        )

    key_array = (
        loan_ids + "|" + periods
    ).to_numpy()

    if len(key_array) > 0:
        if previous_key is not None:
            duplicate_loan_months += int(
                key_array[0] == previous_key
            )

            sort_order_violations += int(
                key_array[0] < previous_key
            )

        if len(key_array) > 1:
            duplicate_loan_months += int(
                (key_array[1:] == key_array[:-1]).sum()
            )

            sort_order_violations += int(
                (key_array[1:] < key_array[:-1]).sum()
            )

        previous_key = key_array[-1]

    del chunk, loan_ids, periods, key_array

gc.collect()

elapsed_seconds = round(
    perf_counter() - start_time,
    1
)

full_2006_check = pd.Series({
    "performance_rows": total_rows,
    "unique_performance_ids": len(
        unique_performance_ids
    ),
    "blank_loan_ids": blank_loan_ids,
    "invalid_period_formats": invalid_period_formats,
    "performance_rows_not_in_originations": (
        unmatched_performance_rows
    ),
    "unique_unmatched_ids": len(unmatched_ids),
    "duplicate_loan_months": duplicate_loan_months,
    "sort_order_violations": sort_order_violations,
    "minimum_reporting_period": minimum_period,
    "maximum_reporting_period": maximum_period,
    "elapsed_seconds": elapsed_seconds
})

full_2006_check

performance_rows                        3203499
unique_performance_ids                    50000
blank_loan_ids                                0
invalid_period_formats                        0
performance_rows_not_in_originations          0
unique_unmatched_ids                          0
duplicate_loan_months                         0
sort_order_violations                         0
minimum_reporting_period                 200601
maximum_reporting_period                 202603
elapsed_seconds                            53.5
dtype: object

In [6]:
def audit_performance_keys(
    vintage,
    file_path,
    valid_origination_ids,
    chunk_size=100_000
):
    total_rows = 0
    unique_ids = set()
    unmatched_ids = set()

    blank_ids = 0
    invalid_periods = 0
    unmatched_rows = 0
    duplicate_keys = 0
    order_violations = 0

    minimum_period = None
    maximum_period = None
    previous_key = None

    start_time = perf_counter()

    for chunk in pd.read_csv(
        file_path,
        sep="|",
        header=None,
        names=performance_columns,
        usecols=[
            "loan_identifier",
            "monthly_reporting_period"
        ],
        dtype="string",
        na_filter=False,
        chunksize=chunk_size
    ):
        loan_ids = (
            chunk["loan_identifier"]
            .str.strip()
        )

        periods = (
            chunk["monthly_reporting_period"]
            .str.strip()
        )

        total_rows += len(chunk)

        blank_mask = loan_ids.eq("")
        invalid_period_mask = (
            ~periods.str.fullmatch(r"\d{6}")
        )
        unmatched_mask = (
            ~loan_ids.isin(valid_origination_ids)
            & ~blank_mask
        )

        blank_ids += int(blank_mask.sum())
        invalid_periods += int(
            invalid_period_mask.sum()
        )
        unmatched_rows += int(
            unmatched_mask.sum()
        )

        unique_ids.update(
            loan_ids.loc[
                ~blank_mask
            ].unique().tolist()
        )

        unmatched_ids.update(
            loan_ids.loc[
                unmatched_mask
            ].unique().tolist()
        )

        valid_periods = periods.loc[
            ~invalid_period_mask
        ]

        if not valid_periods.empty:
            chunk_minimum = valid_periods.min()
            chunk_maximum = valid_periods.max()

            minimum_period = (
                chunk_minimum
                if minimum_period is None
                else min(
                    minimum_period,
                    chunk_minimum
                )
            )

            maximum_period = (
                chunk_maximum
                if maximum_period is None
                else max(
                    maximum_period,
                    chunk_maximum
                )
            )

        key_array = (
            loan_ids + "|" + periods
        ).to_numpy()

        if len(key_array) > 0:
            if previous_key is not None:
                duplicate_keys += int(
                    key_array[0] == previous_key
                )

                order_violations += int(
                    key_array[0] < previous_key
                )

            if len(key_array) > 1:
                duplicate_keys += int(
                    (
                        key_array[1:]
                        == key_array[:-1]
                    ).sum()
                )

                order_violations += int(
                    (
                        key_array[1:]
                        < key_array[:-1]
                    ).sum()
                )

            previous_key = key_array[-1]

        del chunk, loan_ids, periods, key_array

    gc.collect()

    return {
        "vintage": vintage,
        "performance_rows": total_rows,
        "unique_performance_ids": len(unique_ids),
        "blank_loan_ids": blank_ids,
        "invalid_period_formats": invalid_periods,
        "performance_rows_not_in_originations": (
            unmatched_rows
        ),
        "unique_unmatched_ids": len(unmatched_ids),
        "duplicate_loan_months": duplicate_keys,
        "sort_order_violations": order_violations,
        "minimum_reporting_period": minimum_period,
        "maximum_reporting_period": maximum_period,
        "elapsed_seconds": round(
            perf_counter() - start_time,
            1
        )
    }

In [7]:
result_2006 = {
    "vintage": 2006,
    **full_2006_check.to_dict()
}

remaining_results = []

for year in [2015, 2016, 2017]:
    print(f"Auditing {year} performance file...")

    result = audit_performance_keys(
        vintage=year,
        file_path=performance_files[year],
        valid_origination_ids=origination_ids,
        chunk_size=100_000
    )

    remaining_results.append(result)

    print(
        f"{year} complete: "
        f"{result['performance_rows']:,} rows "
        f"in {result['elapsed_seconds']} seconds"
    )

full_key_audit = pd.DataFrame(
    [result_2006] + remaining_results
)

full_key_audit

Auditing 2015 performance file...
2015 complete: 3,467,166 rows in 73.8 seconds
Auditing 2016 performance file...
2016 complete: 3,379,650 rows in 53.0 seconds
Auditing 2017 performance file...
2017 complete: 2,800,219 rows in 47.5 seconds


,vintage,performance_rows,unique_performance_ids,blank_loan_ids,invalid_period_formats,performance_rows_not_in_originations,unique_unmatched_ids,duplicate_loan_months,sort_order_violations,minimum_reporting_period,maximum_reporting_period,elapsed_seconds
0,2006,3203499,50000,0,0,0,0,0,0,200601,202603,53.5
1,2015,3467166,50000,0,0,0,0,0,0,201501,202603,73.8
2,2016,3379650,50000,0,0,0,0,0,0,201601,202603,53.0
3,2017,2800219,50000,0,0,0,0,0,0,201701,202603,47.5


In [8]:
key_control_conclusion = pd.Series({
    "total_performance_rows": (
        full_key_audit["performance_rows"].sum()
    ),
    "total_unique_vintage_loan_ids": (
        full_key_audit[
            "unique_performance_ids"
        ].sum()
    ),
    "blank_loan_ids": (
        full_key_audit["blank_loan_ids"].sum()
    ),
    "invalid_period_formats": (
        full_key_audit[
            "invalid_period_formats"
        ].sum()
    ),
    "unmatched_performance_rows": (
        full_key_audit[
            "performance_rows_not_in_originations"
        ].sum()
    ),
    "unique_unmatched_ids": (
        full_key_audit[
            "unique_unmatched_ids"
        ].sum()
    ),
    "duplicate_loan_months": (
        full_key_audit[
            "duplicate_loan_months"
        ].sum()
    ),
    "sort_order_violations": (
        full_key_audit[
            "sort_order_violations"
        ].sum()
    ),
    "all_key_controls_passed": (
        full_key_audit[
            [
                "blank_loan_ids",
                "invalid_period_formats",
                "performance_rows_not_in_originations",
                "unique_unmatched_ids",
                "duplicate_loan_months",
                "sort_order_violations"
            ]
        ]
        .eq(0)
        .all()
        .all()
    )
})

key_control_conclusion

total_performance_rows           12850534
total_unique_vintage_loan_ids      200000
blank_loan_ids                          0
invalid_period_formats                  0
unmatched_performance_rows              0
unique_unmatched_ids                    0
duplicate_loan_months                   0
sort_order_violations                   0
all_key_controls_passed              True
dtype: object

## Loan-key and population control conclusion

Full-population testing covered 12,850,534 monthly performance records across
the 2006, 2015, 2016, and 2017 vintages.

Results:

- All 200,000 sampled-vintage loans were represented.
- No performance records contained blank loan identifiers.
- Every performance record matched a valid origination loan.
- No invalid monthly reporting-period formats were identified.
- No duplicate loan-month records were identified.
- No file-order violations were identified.

EIT-04 and EIT-09 passed. Scope limitation L-01 is resolved because the
earlier 100,000-row sample was replaced by complete population testing.

In [11]:
import time
import pandas as pd


def audit_performance_values(
    vintage,
    file_path,
    chunk_size=100_000
):
    """
    Audit selected performance-file values without loading
    the complete file into memory.
    """

    columns_needed = [
        "current_actual_upb",
        "current_loan_delinquency_status",
        "loan_age",
        "remaining_months_to_legal_maturity",
        "modification_flag",
        "zero_balance_code",
        "zero_balance_effective_date",
        "due_date_of_last_paid_installment",
    ]

    allowed_modification_flags = {
        "",
        "Y",
        "P",
    }

    allowed_zero_balance_codes = {
        "",
        "01",
        "02",
        "03",
        "09",
        "15",
        "16",
        "96",
    }

    total_rows = 0

    invalid_delinquency_statuses = 0
    ninety_plus_delinquency_rows = 0
    reo_delinquency_rows = 0
    unavailable_delinquency_rows = 0

    invalid_modification_flags = 0
    invalid_zero_balance_codes = 0
    invalid_zero_balance_dates = 0
    invalid_due_dates = 0

    blank_current_upb_rows = 0
    nonnumeric_current_upb_rows = 0
    negative_current_upb_rows = 0

    minimum_loan_age = None
    maximum_loan_age = None
    minimum_remaining_maturity = None
    maximum_remaining_maturity = None

    delinquency_counts = {}

    start_time = time.time()

    reader = pd.read_csv(
        file_path,
        sep="|",
        header=None,
        names=performance_columns,
        usecols=columns_needed,
        dtype="string",
        chunksize=chunk_size,
        low_memory=False,
    )

    for chunk in reader:
        total_rows += len(chunk)

        # Standardize text columns.
        delinquency = (
            chunk["current_loan_delinquency_status"]
            .fillna("")
            .str.strip()
            .str.upper()
        )

        modification = (
            chunk["modification_flag"]
            .fillna("")
            .str.strip()
            .str.upper()
        )

        zero_balance = (
            chunk["zero_balance_code"]
            .fillna("")
            .str.strip()
        )

        zero_balance_date = (
            chunk["zero_balance_effective_date"]
            .fillna("")
            .str.strip()
        )

        due_date = (
            chunk["due_date_of_last_paid_installment"]
            .fillna("")
            .str.strip()
        )

        current_upb_text = (
            chunk["current_actual_upb"]
            .fillna("")
            .str.strip()
        )

        # -----------------------------------------------------
        # Delinquency-status tests
        # -----------------------------------------------------

        valid_delinquency = delinquency.str.fullmatch(
            r"(?:\d{2}|RA|XX)",
            na=False,
        )

        invalid_delinquency_statuses += int(
            (~valid_delinquency).sum()
        )

        numeric_delinquency = pd.to_numeric(
            delinquency.where(
                delinquency.str.fullmatch(r"\d{2}", na=False)
            ),
            errors="coerce",
        )

        # Status 03 or higher represents 90+ days delinquent.
        ninety_plus_delinquency_rows += int(
            numeric_delinquency.ge(3).sum()
        )

        reo_delinquency_rows += int(
            delinquency.eq("RA").sum()
        )

        unavailable_delinquency_rows += int(
            delinquency.eq("XX").sum()
        )

        chunk_status_counts = delinquency.value_counts(
            dropna=False
        )

        for status, count in chunk_status_counts.items():
            status_label = status if status != "" else "<blank>"

            delinquency_counts[status_label] = (
                delinquency_counts.get(status_label, 0)
                + int(count)
            )

        # -----------------------------------------------------
        # Modification and zero-balance category tests
        # -----------------------------------------------------

        invalid_modification_flags += int(
            (
                ~modification.isin(
                    allowed_modification_flags
                )
            ).sum()
        )

        invalid_zero_balance_codes += int(
            (
                ~zero_balance.isin(
                    allowed_zero_balance_codes
                )
            ).sum()
        )

        # -----------------------------------------------------
        # Date-format tests
        # Blank dates are permitted.
        # Nonblank dates must contain six digits: YYYYMM.
        # -----------------------------------------------------

        invalid_zero_balance_dates += int(
            (
                zero_balance_date.ne("")
                & ~zero_balance_date.str.fullmatch(
                    r"\d{6}",
                    na=False,
                )
            ).sum()
        )

        invalid_due_dates += int(
            (
                due_date.ne("")
                & ~due_date.str.fullmatch(
                    r"\d{6}",
                    na=False,
                )
            ).sum()
        )

        # -----------------------------------------------------
        # Current UPB tests
        # -----------------------------------------------------

        blank_current_upb_rows += int(
            current_upb_text.eq("").sum()
        )

        current_upb_numeric = pd.to_numeric(
            current_upb_text,
            errors="coerce",
        )

        nonnumeric_current_upb_rows += int(
            (
                current_upb_text.ne("")
                & current_upb_numeric.isna()
            ).sum()
        )

        negative_current_upb_rows += int(
            current_upb_numeric.lt(0).sum()
        )

        # -----------------------------------------------------
        # Loan-age and remaining-maturity ranges
        # -----------------------------------------------------

        loan_age_numeric = pd.to_numeric(
            chunk["loan_age"],
            errors="coerce",
        )

        remaining_maturity_numeric = pd.to_numeric(
            chunk["remaining_months_to_legal_maturity"],
            errors="coerce",
        )

        chunk_minimum_loan_age = loan_age_numeric.min()
        chunk_maximum_loan_age = loan_age_numeric.max()

        chunk_minimum_remaining = (
            remaining_maturity_numeric.min()
        )

        chunk_maximum_remaining = (
            remaining_maturity_numeric.max()
        )

        if pd.notna(chunk_minimum_loan_age):
            if minimum_loan_age is None:
                minimum_loan_age = chunk_minimum_loan_age
            else:
                minimum_loan_age = min(
                    minimum_loan_age,
                    chunk_minimum_loan_age,
                )

        if pd.notna(chunk_maximum_loan_age):
            if maximum_loan_age is None:
                maximum_loan_age = chunk_maximum_loan_age
            else:
                maximum_loan_age = max(
                    maximum_loan_age,
                    chunk_maximum_loan_age,
                )

        if pd.notna(chunk_minimum_remaining):
            if minimum_remaining_maturity is None:
                minimum_remaining_maturity = (
                    chunk_minimum_remaining
                )
            else:
                minimum_remaining_maturity = min(
                    minimum_remaining_maturity,
                    chunk_minimum_remaining,
                )

        if pd.notna(chunk_maximum_remaining):
            if maximum_remaining_maturity is None:
                maximum_remaining_maturity = (
                    chunk_maximum_remaining
                )
            else:
                maximum_remaining_maturity = max(
                    maximum_remaining_maturity,
                    chunk_maximum_remaining,
                )

    elapsed_seconds = round(
        time.time() - start_time,
        1,
    )

    summary_result = {
        "vintage": vintage,
        "performance_rows": total_rows,
        "invalid_delinquency_statuses": (
            invalid_delinquency_statuses
        ),
        "ninety_plus_delinquency_rows": (
            ninety_plus_delinquency_rows
        ),
        "reo_delinquency_rows": reo_delinquency_rows,
        "unavailable_delinquency_rows": (
            unavailable_delinquency_rows
        ),
        "invalid_modification_flags": (
            invalid_modification_flags
        ),
        "invalid_zero_balance_codes": (
            invalid_zero_balance_codes
        ),
        "invalid_zero_balance_dates": (
            invalid_zero_balance_dates
        ),
        "invalid_due_dates": invalid_due_dates,
        "blank_current_upb_rows": blank_current_upb_rows,
        "nonnumeric_current_upb_rows": (
            nonnumeric_current_upb_rows
        ),
        "negative_current_upb_rows": (
            negative_current_upb_rows
        ),
        "minimum_loan_age": minimum_loan_age,
        "maximum_loan_age": maximum_loan_age,
        "minimum_remaining_maturity": (
            minimum_remaining_maturity
        ),
        "maximum_remaining_maturity": (
            maximum_remaining_maturity
        ),
        "elapsed_seconds": elapsed_seconds,
    }

    status_rows = pd.DataFrame(
        [
            {
                "vintage": vintage,
                "delinquency_status": status,
                "record_count": count,
            }
            for status, count in delinquency_counts.items()
        ]
    )

    return summary_result, status_rows

In [12]:
vintages = [2006, 2015, 2016, 2017]

value_audit_results = []
delinquency_status_results = []

for year in vintages:
    print(f"Auditing {year} values...")

    summary_result, status_rows = (
        audit_performance_values(
            vintage=year,
            file_path=performance_files[year],
            chunk_size=100_000,
        )
    )

    value_audit_results.append(summary_result)
    delinquency_status_results.append(status_rows)

    print(
        f"{year} complete: "
        f"{summary_result['performance_rows']:,} rows "
        f"in {summary_result['elapsed_seconds']} seconds"
    )

performance_value_audit = pd.DataFrame(
    value_audit_results
)

delinquency_status_counts = pd.concat(
    delinquency_status_results,
    ignore_index=True,
)

top_delinquency_statuses = (
    delinquency_status_counts
    .sort_values(
        ["vintage", "record_count"],
        ascending=[True, False],
    )
    .groupby("vintage")
    .head(10)
    .reset_index(drop=True)
)

display(performance_value_audit)
display(top_delinquency_statuses)

Auditing 2006 values...
2006 complete: 3,203,499 rows in 17.1 seconds
Auditing 2015 values...
2015 complete: 3,467,166 rows in 13.6 seconds
Auditing 2016 values...
2016 complete: 3,379,650 rows in 13.9 seconds
Auditing 2017 values...
2017 complete: 2,800,219 rows in 12.0 seconds


,vintage,performance_rows,invalid_delinquency_statuses,ninety_plus_delinquency_rows,reo_delinquency_rows,unavailable_delinquency_rows,invalid_modification_flags,invalid_zero_balance_codes,invalid_zero_balance_dates,invalid_due_dates,blank_current_upb_rows,nonnumeric_current_upb_rows,negative_current_upb_rows,minimum_loan_age,maximum_loan_age,minimum_remaining_maturity,maximum_remaining_maturity,elapsed_seconds
0,2006,3203499,0,125626,19437,0,0,0,0,0,0,0,0,0,242,-8,517,17.1
1,2015,3467166,0,20958,567,0,0,0,0,0,0,0,0,0,134,-1,480,13.6
2,2016,3379650,0,22262,216,0,0,0,0,0,0,0,0,0,122,1,480,13.9
3,2017,2800219,0,30072,352,0,0,0,0,0,0,0,0,0,110,4,480,12.0


,vintage,delinquency_status,record_count
0,2006,00,2940779
1,2006,01,87217
2,2006,02,30440
3,2006,RA,19437
4,2006,03,15291
5,2006,04,11617
6,2006,05,9785
7,2006,06,8346
8,2006,07,7318
9,2006,08,6549


## Performance Value and Category Testing Conclusion

Full-population, memory-safe testing was completed across 12,850,534
monthly performance records for the 2006, 2015, 2016, and 2017
sample vintages.

No exceptions were identified for:

- delinquency-status validity;
- modification-flag validity;
- zero-balance-code validity;
- tested date formats;
- blank or nonnumeric Current Actual UPB; or
- negative Current Actual UPB.

The 2006 vintage contained substantially more 90+ delinquent and REO
records than the post-crisis vintages. These records represent observed
credit outcomes and are not data-quality exceptions.

Remaining months to legal maturity included values below zero and above
480 months. These are recorded as observations requiring targeted
follow-up because modified loans use the modified maturity date in the
calculation. They are not classified as confirmed data defects at this
stage.

**Preliminary conclusion:** Performance value and category controls
passed, subject to targeted review of remaining-maturity extremes.

In [13]:
maturity_audit_results = []
maturity_sample_results = []

columns_needed = [
    "loan_identifier",
    "monthly_reporting_period",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "modification_flag",
    "zero_balance_code",
]

for year in vintages:
    print(f"Reviewing {year} maturity extremes...")

    negative_rows = 0
    above_480_rows = 0
    modified_outlier_rows = 0
    zero_balance_outlier_rows = 0

    negative_loan_ids = set()
    above_480_loan_ids = set()

    vintage_samples = []

    reader = pd.read_csv(
        performance_files[year],
        sep="|",
        header=None,
        names=performance_columns,
        usecols=columns_needed,
        dtype="string",
        chunksize=100_000,
        low_memory=False,
    )

    for chunk in reader:
        remaining_maturity = pd.to_numeric(
            chunk["remaining_months_to_legal_maturity"],
            errors="coerce",
        )

        modification = (
            chunk["modification_flag"]
            .fillna("")
            .str.strip()
            .str.upper()
        )

        zero_balance = (
            chunk["zero_balance_code"]
            .fillna("")
            .str.strip()
        )

        negative_mask = remaining_maturity.lt(0)
        above_480_mask = remaining_maturity.gt(480)
        outlier_mask = negative_mask | above_480_mask

        negative_rows += int(negative_mask.sum())
        above_480_rows += int(above_480_mask.sum())

        negative_loan_ids.update(
            chunk.loc[
                negative_mask,
                "loan_identifier",
            ].dropna()
        )

        above_480_loan_ids.update(
            chunk.loc[
                above_480_mask,
                "loan_identifier",
            ].dropna()
        )

        modified_outlier_rows += int(
            (
                outlier_mask
                & modification.isin(["Y", "P"])
            ).sum()
        )

        zero_balance_outlier_rows += int(
            (
                outlier_mask
                & zero_balance.ne("")
            ).sum()
        )

        # Retain only a small sample for documentation.
        if outlier_mask.any() and len(vintage_samples) < 20:
            available_spaces = 20 - len(vintage_samples)

            sample = (
                chunk.loc[outlier_mask, columns_needed]
                .head(available_spaces)
                .copy()
            )

            sample.insert(0, "vintage", year)

            vintage_samples.extend(
                sample.to_dict("records")
            )

    total_outlier_rows = (
        negative_rows + above_480_rows
    )

    maturity_audit_results.append(
        {
            "vintage": year,
            "negative_maturity_rows": negative_rows,
            "negative_maturity_loans": len(
                negative_loan_ids
            ),
            "above_480_maturity_rows": above_480_rows,
            "above_480_maturity_loans": len(
                above_480_loan_ids
            ),
            "total_outlier_rows": total_outlier_rows,
            "modified_outlier_rows": modified_outlier_rows,
            "zero_balance_outlier_rows": (
                zero_balance_outlier_rows
            ),
        }
    )

    maturity_sample_results.extend(vintage_samples)

    print(
        f"{year} complete: "
        f"{total_outlier_rows:,} outlier rows"
    )

maturity_extreme_summary = pd.DataFrame(
    maturity_audit_results
)

maturity_extreme_samples = pd.DataFrame(
    maturity_sample_results
)

display(maturity_extreme_summary)
display(maturity_extreme_samples)

Reviewing 2006 maturity extremes...
2006 complete: 63 outlier rows
Reviewing 2015 maturity extremes...
2015 complete: 1 outlier rows
Reviewing 2016 maturity extremes...
2016 complete: 0 outlier rows
Reviewing 2017 maturity extremes...
2017 complete: 0 outlier rows


,vintage,negative_maturity_rows,negative_maturity_loans,above_480_maturity_rows,above_480_maturity_loans,total_outlier_rows,modified_outlier_rows,zero_balance_outlier_rows
0,2006,26,8,37,1,63,9,7
1,2015,1,1,0,0,1,0,1
2,2016,0,0,0,0,0,0,0
3,2017,0,0,0,0,0,0,0


,vintage,loan_identifier,monthly_reporting_period,loan_age,remaining_months_to_legal_maturity,modification_flag,zero_balance_code
0,2006,F06Q10030469,201803,61,-1,P,NaN
1,2006,F06Q10030469,201804,62,-2,P,NaN
2,2006,F06Q10030469,201805,63,-3,P,NaN
3,2006,F06Q10030469,201806,64,-4,P,NaN
4,2006,F06Q10030469,201807,65,-5,P,NaN
5,2006,F06Q10030469,201808,66,-6,P,NaN
6,2006,F06Q10030469,201809,67,-7,P,NaN
7,2006,F06Q10030469,201810,68,-8,P,01
8,2006,F06Q10051969,201603,121,-1,NaN,01
9,2006,F06Q20260586,202108,181,-1,NaN,01


In [14]:
from pathlib import Path
import pyarrow.parquet as pq

# Locate the cleaned origination file.
possible_origination_paths = [
    Path("data/processed/originations_clean.parquet"),
    Path("../data/processed/originations_clean.parquet"),
]

origination_parquet_path = next(
    (
        path
        for path in possible_origination_paths
        if path.exists()
    ),
    None,
)

if origination_parquet_path is None:
    raise FileNotFoundError(
        "Could not locate originations_clean.parquet."
    )

all_maturity_outliers = []

columns_needed = [
    "loan_identifier",
    "monthly_reporting_period",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "modification_flag",
    "zero_balance_code",
]

for year in vintages:
    print(f"Collecting {year} maturity outliers...")

    reader = pd.read_csv(
        performance_files[year],
        sep="|",
        header=None,
        names=performance_columns,
        usecols=columns_needed,
        dtype="string",
        chunksize=100_000,
        low_memory=False,
    )

    for chunk in reader:
        remaining_maturity = pd.to_numeric(
            chunk["remaining_months_to_legal_maturity"],
            errors="coerce",
        )

        outlier_mask = (
            remaining_maturity.lt(0)
            | remaining_maturity.gt(480)
        )

        if outlier_mask.any():
            outliers = chunk.loc[
                outlier_mask,
                columns_needed,
            ].copy()

            outliers.insert(0, "vintage", year)

            all_maturity_outliers.append(outliers)

maturity_outlier_records = pd.concat(
    all_maturity_outliers,
    ignore_index=True,
)

# Read only the necessary origination columns.
parquet_columns = (
    pq.ParquetFile(origination_parquet_path)
    .schema.names
)

desired_origination_columns = [
    "loan_identifier",
    "first_payment_date",
    "maturity_date",
    "original_loan_term",
]

available_origination_columns = [
    column
    for column in desired_origination_columns
    if column in parquet_columns
]

origination_terms = pd.read_parquet(
    origination_parquet_path,
    columns=available_origination_columns,
)

maturity_outlier_records = (
    maturity_outlier_records.merge(
        origination_terms,
        on="loan_identifier",
        how="left",
        validate="many_to_one",
    )
)

maturity_outlier_records["is_modified"] = (
    maturity_outlier_records["modification_flag"]
    .fillna("")
    .str.strip()
    .str.upper()
    .isin(["Y", "P"])
)

maturity_outlier_records["has_zero_balance_code"] = (
    maturity_outlier_records["zero_balance_code"]
    .fillna("")
    .str.strip()
    .ne("")
)

maturity_outlier_records[
    "remaining_months_to_legal_maturity"
] = pd.to_numeric(
    maturity_outlier_records[
        "remaining_months_to_legal_maturity"
    ],
    errors="coerce",
)

maturity_outlier_loan_summary = (
    maturity_outlier_records
    .groupby(
        ["vintage", "loan_identifier"],
        as_index=False,
    )
    .agg(
        outlier_rows=(
            "remaining_months_to_legal_maturity",
            "size",
        ),
        minimum_remaining_maturity=(
            "remaining_months_to_legal_maturity",
            "min",
        ),
        maximum_remaining_maturity=(
            "remaining_months_to_legal_maturity",
            "max",
        ),
        first_outlier_period=(
            "monthly_reporting_period",
            "min",
        ),
        last_outlier_period=(
            "monthly_reporting_period",
            "max",
        ),
        modified_outlier_rows=(
            "is_modified",
            "sum",
        ),
        zero_balance_outlier_rows=(
            "has_zero_balance_code",
            "sum",
        ),
        first_payment_date=(
            "first_payment_date",
            "first",
        ),
        maturity_date=(
            "maturity_date",
            "first",
        ),
        original_loan_term=(
            "original_loan_term",
            "first",
        ),
    )
    .sort_values(
        ["vintage", "loan_identifier"]
    )
    .reset_index(drop=True)
)

print(
    "Outlier records:",
    f"{len(maturity_outlier_records):,}",
)

print(
    "Affected loans:",
    maturity_outlier_records[
        "loan_identifier"
    ].nunique(),
)

display(maturity_outlier_loan_summary)

Outlier records: 64
Affected loans: 10


,vintage,loan_identifier,outlier_rows,minimum_remaining_maturity,maximum_remaining_maturity,first_outlier_period,last_outlier_period,modified_outlier_rows,zero_balance_outlier_rows,first_payment_date,maturity_date,original_loan_term
0,2006,F06Q10030469,8,-8,-1,201803,201810,8,1,2006-03-01,2018-02-01,144
1,2006,F06Q10051969,1,-1,-1,201603,201603,0,1,2006-03-01,2016-02-01,120
2,2006,F06Q20260586,1,-1,-1,202108,202108,0,1,2006-08-01,2021-07-01,180
3,2006,F06Q30272216,1,-1,-1,201903,201903,1,1,2006-11-01,2036-10-01,360
4,2006,F06Q30294070,37,481,517,200610,200910,0,0,2006-11-01,2036-10-01,360
5,2006,F06Q30323261,7,-7,-1,202110,202204,0,1,2006-10-01,2021-09-01,180
6,2006,F06Q40097512,1,-1,-1,202201,202201,0,1,2007-01-01,2021-12-01,180
7,2006,F06Q40399461,1,-1,-1,202202,202202,0,1,2007-02-01,2022-01-01,180
8,2006,F06Q40412935,6,-6,-1,202202,202207,0,0,2007-02-01,2022-01-01,180
9,2015,F15Q30002246,1,-1,-1,202509,202509,0,1,2015-09-01,2025-08-01,120


In [15]:
maturity_reconciliation = (
    maturity_outlier_records.copy()
)

# Convert reporting period to a date.
maturity_reconciliation["reporting_date"] = (
    pd.to_datetime(
        maturity_reconciliation[
            "monthly_reporting_period"
        ],
        format="%Y%m",
        errors="coerce",
    )
)

# Ensure origination maturity is a date.
maturity_reconciliation["maturity_date"] = (
    pd.to_datetime(
        maturity_reconciliation["maturity_date"],
        errors="coerce",
    )
)

# Calculate remaining months using the original maturity date.
maturity_reconciliation[
    "calculated_original_remaining_months"
] = (
    (
        maturity_reconciliation["maturity_date"].dt.year
        - maturity_reconciliation["reporting_date"].dt.year
    )
    * 12
    + (
        maturity_reconciliation["maturity_date"].dt.month
        - maturity_reconciliation["reporting_date"].dt.month
    )
)

# Compare the disclosed value with the calculated value.
maturity_reconciliation[
    "reported_minus_calculated"
] = (
    maturity_reconciliation[
        "remaining_months_to_legal_maturity"
    ]
    - maturity_reconciliation[
        "calculated_original_remaining_months"
    ]
)

maturity_reconciliation_summary = (
    maturity_reconciliation
    .groupby(
        ["vintage", "loan_identifier"],
        as_index=False,
    )
    .agg(
        outlier_rows=(
            "remaining_months_to_legal_maturity",
            "size",
        ),
        minimum_reported_maturity=(
            "remaining_months_to_legal_maturity",
            "min",
        ),
        maximum_reported_maturity=(
            "remaining_months_to_legal_maturity",
            "max",
        ),
        minimum_calculated_maturity=(
            "calculated_original_remaining_months",
            "min",
        ),
        maximum_calculated_maturity=(
            "calculated_original_remaining_months",
            "max",
        ),
        minimum_difference=(
            "reported_minus_calculated",
            "min",
        ),
        maximum_difference=(
            "reported_minus_calculated",
            "max",
        ),
        modified_rows=(
            "is_modified",
            "sum",
        ),
        zero_balance_rows=(
            "has_zero_balance_code",
            "sum",
        ),
    )
    .sort_values(
        ["vintage", "loan_identifier"]
    )
    .reset_index(drop=True)
)

display(maturity_reconciliation_summary)

,vintage,loan_identifier,outlier_rows,minimum_reported_maturity,maximum_reported_maturity,minimum_calculated_maturity,maximum_calculated_maturity,minimum_difference,maximum_difference,modified_rows,zero_balance_rows
0,2006,F06Q10030469,8,-8,-1,-8,-1,0,0,8,1
1,2006,F06Q10051969,1,-1,-1,-1,-1,0,0,0,1
2,2006,F06Q20260586,1,-1,-1,-1,-1,0,0,0,1
3,2006,F06Q30272216,1,-1,-1,211,211,-212,-212,1,1
4,2006,F06Q30294070,37,481,517,324,360,157,157,0,0
5,2006,F06Q30323261,7,-7,-1,-7,-1,0,0,0,1
6,2006,F06Q40097512,1,-1,-1,-1,-1,0,0,0,1
7,2006,F06Q40399461,1,-1,-1,-1,-1,0,0,0,1
8,2006,F06Q40412935,6,-6,-1,-6,-1,0,0,0,0
9,2015,F15Q30002246,1,-1,-1,-1,-1,0,0,0,1


## Final Remaining-Maturity Review

Targeted follow-up testing was performed on 64 performance records
with remaining maturity below zero or above 480 months.

### Negative maturity observations

Twenty-six negative-maturity records reconciled exactly to the difference
between the original maturity date and monthly reporting period. These
records represent reporting after scheduled maturity and are not
classified as data-quality exceptions.

One additional record belonged to a modified loan and contained a
zero-balance termination code. It could not be independently reconciled
to the original maturity date because Freddie Mac calculates remaining
maturity using the modified maturity date after a modification. The
modified maturity date is not separately available in the sample data.
This item is therefore treated as a scope limitation rather than a
confirmed exception.

### Confirmed isolated exception

Loan `F06Q30294070` contained 37 performance records with reported
remaining maturity between 481 and 517 months. Based on its October
2036 original maturity date, the independently calculated range was
324 to 360 months.

The reported values exceeded the independently calculated values by
exactly 157 months in every affected period. The loan had no applicable
modification flag or zero-balance code explaining the difference.

This represents:

- 1 of 200,000 loans tested (0.0005%);
- 37 of 12,850,534 performance records tested (0.00029%); and
- an isolated, consistently offset field-level discrepancy.

### Control conclusion

The remaining-maturity control is assessed as **effective with an
isolated low-severity exception**. The exception does not indicate a
pervasive population-level control failure, but it is retained for
transparent reporting and audit traceability.